In [1]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

np.random.seed(42)

In [2]:
# Reuse a simple automotive price dataset for the pipeline demo
n = 500
engine_size_L = np.random.uniform(1.0, 5.0, n)
horsepower = 60 + engine_size_L * 40 + np.random.normal(0, 15, n)
age_years = np.random.uniform(0, 15, n)
price = 25000 + horsepower * 80 - age_years * 1300 + np.random.normal(0, 1500, n)

df = pd.DataFrame({"engine_size_L": engine_size_L, "horsepower": horsepower,
                    "age_years": age_years, "price": price})

X_train, X_test, y_train, y_test = train_test_split(
    df.drop(columns=["price"]), df["price"], test_size=0.2, random_state=42)

print("Data ready:", X_train.shape, X_test.shape)

Data ready: (400, 3) (100, 3)


In [4]:
!pip install mlflow

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.0/50.0 kB 1.4 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.9/50.9 kB 4.2 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.2/44.2 kB 3.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.7/11.7 MB 66.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.8/3.8 MB 100.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 62.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 268.7/268.7 kB 19.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 148.8/148.8 kB 12.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 114.9/114.9 kB 9.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 228.4/228.4 kB 17.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 136.5/136.5 kB 11.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 144.6/144.6 kB 10.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

In [7]:
import mlflow
import mlflow.sklearn

mlflow.set_tracking_uri("sqlite:///mlflow.db")   # local tracking store (swap for a remote server in production)
mlflow.set_experiment("vehicle_price_ci_cd")

models_to_try = {
    "linear_regression": LinearRegression(),
    "random_forest": RandomForestRegressor(n_estimators=200, random_state=42),
}

run_ids = {}

for name, model in models_to_try.items():
    with mlflow.start_run(run_name=name) as run:
        model.fit(X_train, y_train)
        preds = model.predict(X_test)

        mae = mean_absolute_error(y_test, preds)
        rmse = mean_squared_error(y_test, preds) ** 0.5
        r2 = r2_score(y_test, preds)

        mlflow.log_param("model_type", name)
        mlflow.log_metric("mae", mae)
        mlflow.log_metric("rmse", rmse)
        mlflow.log_metric("r2", r2)
        mlflow.sklearn.log_model(
            model,
            name="model",
            skops_trusted_types=["sklearn.tree._tree.Tree"]
        )

        run_ids[name] = run.info.run_id
        print(f"{name}: MAE={mae:.2f} RMSE={rmse:.2f} R2={r2:.3f} (run_id={run.info.run_id})")

linear_regression: MAE=1435.56 RMSE=1768.40 R2=0.940 (run_id=5545322d0ecc481fae5f65df4410c8d8)
random_forest: MAE=1531.31 RMSE=1970.82 R2=0.926 (run_id=a49a62a538094c8aa3fae1768e7b44bc)


In [8]:
# --- Pick the best run from the MLflow experiment and register it ---
client = mlflow.tracking.MlflowClient()
experiment = client.get_experiment_by_name("vehicle_price_ci_cd")
runs = client.search_runs(experiment.experiment_id, order_by=["metrics.rmse ASC"])

best_run = runs[0]
best_model_name = best_run.data.params["model_type"]
print(f"Best model: {best_model_name} | RMSE={best_run.data.metrics['rmse']:.2f}")

model_uri = f"runs:/{best_run.info.run_id}/model"
registered = mlflow.register_model(model_uri, "vehicle_price_predictor")
print(f"Registered model '{registered.name}' version {registered.version}")


Successfully registered model 'vehicle_price_predictor'.
2026/09/23 15:32:18 WARNING mlflow.tracking._model_registry.fluent: Run with id 5545322d0ecc481fae5f65df4410c8d8 has no artifacts at artifact path 'model', registering model based on models:/m-11e4eebdce6f4effa359b8f2065e4668 instead


Best model: linear_regression | RMSE=1768.40
Registered model 'vehicle_price_predictor' version 1


Created version '1' of model 'vehicle_price_predictor'.


In [9]:
# Generate a Dockerfile that serves the registered MLflow model
dockerfile_content = f'''FROM python:3.10-slim

WORKDIR /app

RUN pip install mlflow scikit-learn pandas numpy

COPY mlruns /app/mlruns

ENV MLFLOW_TRACKING_URI=file:/app/mlruns

EXPOSE 5001

CMD ["mlflow", "models", "serve", \\
     "-m", "models:/vehicle_price_predictor/{registered.version}", \\
     "-h", "0.0.0.0", "-p", "5001", "--no-conda"]
'''

with open("Dockerfile", "w") as f:
    f.write(dockerfile_content)

print(dockerfile_content)


FROM python:3.10-slim

WORKDIR /app

RUN pip install mlflow scikit-learn pandas numpy

COPY mlruns /app/mlruns

ENV MLFLOW_TRACKING_URI=file:/app/mlruns

EXPOSE 5001

CMD ["mlflow", "models", "serve", \
     "-m", "models:/vehicle_price_predictor/1", \
     "-h", "0.0.0.0", "-p", "5001", "--no-conda"]



In [10]:
# Minimal CI/CD pipeline definition (GitHub Actions) simulating the deploy workflow
ci_cd_yaml = '''name: ml-ci-cd-pipeline

on:
  push:
    branches: [main]

jobs:
  train-and-deploy:
    runs-on: ubuntu-latest
    steps:
      - uses: actions/checkout@v4

      - name: Set up Python
        uses: actions/setup-python@v5
        with:
          python-version: "3.10"

      - name: Install dependencies
        run: pip install -r requirements.txt

      - name: Run training + MLflow logging
        run: python train.py

      - name: Build Docker image
        run: docker build -t vehicle-price-model:latest .

      - name: Push Docker image
        run: |
          docker tag vehicle-price-model:latest myregistry/vehicle-price-model:latest
          docker push myregistry/vehicle-price-model:latest

      - name: Deploy container
        run: docker run -d -p 5001:5001 myregistry/vehicle-price-model:latest
'''

with open("ci_cd_pipeline.yaml", "w") as f:
    f.write(ci_cd_yaml)

print(ci_cd_yaml)
print("\nPipeline simulation complete: train -> track (MLflow) -> register -> containerize (Docker) -> deploy (CI/CD).")


name: ml-ci-cd-pipeline

on:
  push:
    branches: [main]

jobs:
  train-and-deploy:
    runs-on: ubuntu-latest
    steps:
      - uses: actions/checkout@v4

      - name: Set up Python
        uses: actions/setup-python@v5
        with:
          python-version: "3.10"

      - name: Install dependencies
        run: pip install -r requirements.txt

      - name: Run training + MLflow logging
        run: python train.py

      - name: Build Docker image
        run: docker build -t vehicle-price-model:latest .

      - name: Push Docker image
        run: |
          docker tag vehicle-price-model:latest myregistry/vehicle-price-model:latest
          docker push myregistry/vehicle-price-model:latest

      - name: Deploy container
        run: docker run -d -p 5001:5001 myregistry/vehicle-price-model:latest


Pipeline simulation complete: train -> track (MLflow) -> register -> containerize (Docker) -> deploy (CI/CD).
